In [11]:
import pandas as pd
import matplotlib.pyplot as plt
from raspberry_connector import RaspberryConnector

Connected to localhost with success.


In [ ]:
rc = RaspberryConnector(conf_path="conf.json", devices_path="devices.json")

In [ ]:
temp = rc.temp_sens.value
air_humidity = rc.air_hum_sens.value
pH = rc.pH_meter.value
light = rc.PAR_meter.value
soil_humidity = rc.grodan.value

data_with_timestamps = {
    'timestamp': temp.index.to_list(),
    'temperature': temp.to_list(),
    'air_humidity': air_humidity.to_list(),
    'pH': pH.to_list(),
    'PAR': light.to_list(),
    'soil_humidity': soil_humidity.to_list()
}

df = pd.DataFrame(data_with_timestamps)
df.to_json("mock_data_3.json", orient='records')
print(df)

In [57]:
# Extraxt data from json
df = pd.read_json("mock_data_3.json")
start_date = '2024-01-01'
end_date = '2024-07-01'

# Seleziona i dati dal 1 gennaio al 1 luglio compreso
df = df.loc[(df['timestamp'] >= start_date) & (df['timestamp'] <= end_date)]
df.set_index('timestamp', inplace=True)
df.index.name = None
print(df)

Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
C

In [58]:
# Copia i dati da gennaio a luglio
df_copy = df.copy()

# Inverti l'ordine dei dati copiati
df_copy = df_copy[::-1]

# Genera nuove timestamp della stessa lunghezza del DataFrame copiato
last_timestamp = pd.to_datetime(df.index[-1])
new_timestamps = pd.date_range(start=last_timestamp + pd.Timedelta(seconds=30), 
                               periods=len(df_copy), freq='1S')

# Imposta le nuove timestamp invertite sul DataFrame copiato
df_copy.index = new_timestamps

# Unisci il DataFrame originale con quello invertito
df_final = pd.concat([df, df_copy])

# Mostra il risultato finale
print(df_final)

Connected to localhost with success.
                     temperature  air_humidity   pH  PAR  soil_humidity
2024-01-01 00:00:00        21.00         51.90  6.5  0.0           93.0
2024-01-01 00:00:01        21.00         51.90  6.5  0.0           93.0
2024-01-01 00:00:02        21.00         51.90  6.5  0.0           93.0
2024-01-01 00:00:03        21.00         51.90  6.5  0.0           93.0
2024-01-01 00:00:04        21.01         51.89  6.5  0.0           93.0
...                          ...           ...  ...  ...            ...
2024-12-30 00:00:26        21.01         51.89  6.5  0.0           93.0
2024-12-30 00:00:27        21.00         51.90  6.5  0.0           93.0
2024-12-30 00:00:28        21.00         51.90  6.5  0.0           93.0
2024-12-30 00:00:29        21.00         51.90  6.5  0.0           93.0
2024-12-30 00:00:30        21.00         51.90  6.5  0.0           93.0

[31449602 rows x 5 columns]


In [61]:
# Calcola il numero di righe mancanti per coprire fino al 31 dicembre 23:59:30
last_timestamp = pd.to_datetime(df_final.index[-1])
missing_timestamps = pd.date_range(start=last_timestamp + pd.Timedelta(seconds=30), 
                                   end='2024-12-31 23:59:59', freq='1S')

# Seleziona un singolo campione (ad esempio l'ultimo) per coprire i dati mancanti
last_row_data = df_final.iloc[-1]  # Seleziona l'ultima riga

# Ripeti l'ultimo campione per il numero di timestamp mancanti
missing_data = pd.DataFrame([last_row_data.values] * len(missing_timestamps), 
                            columns=df_final.columns, index=missing_timestamps)

# Unisci i dati mancanti con il DataFrame originale
df_final_complete = pd.concat([df_final, missing_data])

# Mostra il risultato finale
print(df_final_complete)

Connected to localhost with success.
                     temperature  air_humidity   pH  PAR  soil_humidity
2024-01-01 00:00:00        21.00         51.90  6.5  0.0           93.0
2024-01-01 00:00:01        21.00         51.90  6.5  0.0           93.0
2024-01-01 00:00:02        21.00         51.90  6.5  0.0           93.0
2024-01-01 00:00:03        21.00         51.90  6.5  0.0           93.0
2024-01-01 00:00:04        21.01         51.89  6.5  0.0           93.0
...                          ...           ...  ...  ...            ...
2024-12-31 23:59:55        21.00         51.90  6.5  0.0           93.0
2024-12-31 23:59:56        21.00         51.90  6.5  0.0           93.0
2024-12-31 23:59:57        21.00         51.90  6.5  0.0           93.0
2024-12-31 23:59:58        21.00         51.90  6.5  0.0           93.0
2024-12-31 23:59:59        21.00         51.90  6.5  0.0           93.0

[31622342 rows x 5 columns]


In [55]:
# Creazione dei subplots per ogni colonna
fig, axs = plt.subplots(nrows=5, ncols=1, figsize=(10, 15))

# Lista delle colonne per iterazione
columns = ['temperature', 'air_humidity', 'pH', 'PAR', 'soil_humidity']

# Itera sulle colonne e crea un subplot per ciascuna
for i, col in enumerate(columns):
    axs[i].plot(df_final_complete.index, df_final_complete[col], label=col)
    axs[i].set_title(f'{col} over time')
    axs[i].set_xlabel('Time')
    axs[i].set_ylabel(col)
    axs[i].legend()

# Migliora il layout
plt.tight_layout()

# Mostra il grafico
fig_path = "mesures_ab.png"
plt.savefig(fig_path)
plt.close()

Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.


In [62]:
temp = df_final_complete['temperature']
air_humidity = df_final_complete['air_humidity']
pH = df_final_complete['pH']
light = df_final_complete['PAR']
soil_humidity = df_final_complete['soil_humidity']

In [63]:
data_with_timestamps = {
    'timestamp': temp.index.to_list(),
    'temperature': temp.to_list(),
    'air_humidity': air_humidity.to_list(),
    'pH': pH.to_list(),
    'PAR': light.to_list(),
    'soil_humidity': soil_humidity.to_list()
}

df = pd.DataFrame(data_with_timestamps)
df.to_json("mock_data_final.json", orient='records')

Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
C

In [64]:
PAR_meter = {
    'timestamp': temp.index.to_list(),
    'PAR': light.to_list(),
}

df = pd.DataFrame(PAR_meter)
df.to_json("PAR_meter.json", orient='records')

Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.


In [65]:
grodan = {
    'timestamp': temp.index.to_list(),
    'soil_humidity': soil_humidity.to_list()
}

df = pd.DataFrame(grodan)
df.to_json("grodan.json", orient='records')

Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
C

In [66]:
pH_meter = {
    'timestamp': temp.index.to_list(),
    'soil_pH': pH.to_list(),
}

df = pd.DataFrame(pH_meter)
df.to_json("pH_meter.json", orient='records')

Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
C

In [68]:
dht11 = {
    'timestamp': temp.index.to_list(),
    'air_temperature': temp.to_list(),
    'air_humidity': air_humidity.to_list(),
}

df = pd.DataFrame(dht11)
df.to_json("dht11.json", orient='records')

Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
Connected to localhost with success.
C

OverflowError: Could not reserve memory block